# Etapa 2, E4 krok C: väčší model (Qwen2.5-7B-Instruct v 4 bitoch)

Ablácia veľkosti modelu: pomôže väčší model samotnému výberu alebo spojeniu s klasifikátorom, a dokáže využiť počty a príklady z train (prompt v2), ktoré 3B model nevyužil? 7B vo float16 má 15 GB a do 8 GB GPU sa nezmestí, preto 4-bit NF4 (bitsandbytes), výpočty vo float16. Všetko je na **valid**.

| Bunka | Čo robí | Čas |
| --- | --- | --- |
| 1 | nastavenie, kontrola súborov | hneď |
| 2 | inštalácia bitsandbytes a accelerate (a torch/transformers, ak chýbajú) | 1 až 5 min |
| 3 | kontrola GPU a bitsandbytes | pár sekúnd |
| 4 | stiahnutie 7B modelu (asi 5,5 GB, len prvýkrát) | asi 5 min |
| 5 | skúška na 20 správach, meranie rýchlosti | 2 až 4 min |
| 6 | prompt v1 na celom valid | asi 10 až 20 min |
| 7 | prompt v2 na celom valid | asi 15 až 25 min |
| 8 | zbalenie výsledkov do `vysledky_e4c.zip` | hneď |
| 9 | (na konci projektu) zmazanie 7B modelu | hneď |

Spúšťaj bunky postupne zhora (Shift+Enter). Odpovede sa ukladajú priebežne; po prerušení spusti bunky 1, 3 a tú, ktorá bežala, pokračuje sa tam, kde sa skončilo.

**Bunka 1:** 7B model pôjde do podpriečinka `PRIECINOK_MODELOV`, vedľa 3B modelu. Všetky súbory musia byť `OK`.

In [ ]:
import glob, importlib.util, os, shutil, sys, time, zipfile

# >>> nastavenie <<<
PRIECINOK_MODELOV = "/home/jovyan/work/SpracovaniePrirodzenehoJazyka/zadanie_2etapa/llm_modely"
MODEL = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"   # už skvantovaný na 4 bity (asi 5,5 GB)
# záloha, ak by repozitár vyššie nebol dostupný: pôvodný model (15 GB), skvantuje sa pri načítaní
# MODEL = "Qwen/Qwen2.5-7B-Instruct"

MODEL_CESTA = os.path.join(PRIECINOK_MODELOV, MODEL.split("/")[-1])
os.environ["HF_HOME"] = os.path.join(PRIECINOK_MODELOV, "hf_home")
os.environ["PIP_NO_CACHE_DIR"] = "1"
KOREN = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(KOREN)
print("koreň projektu:", KOREN, "| model:", MODEL_CESTA)
potrebne = ["src/spolocne.py", "src/e3_hierarchia.py", "src/e4_llm.py", "data/incidents_train.csv",
            "data/incidents_valid.csv", "results/etapa2/kandidati/E12kens_valid.csv",
            "results/etapa2/predikcie/E12kens_valid.csv", "requirements_e4.txt"]
for s in potrebne:
    print("OK    " if os.path.exists(s) else "CHÝBA ", s)
assert all(os.path.exists(s) for s in potrebne), "niektorý súbor chýba, skontroluj štruktúru priečinkov"

**Bunka 2:** doinštaluje chýbajúce balíčky z `requirements_e4.txt` (nové sú `bitsandbytes` a `accelerate`). Varovanie o `httpcore`/`h11` je neškodné.

In [ ]:
chybaju = [b for b in ("torch", "transformers", "bitsandbytes", "accelerate") if importlib.util.find_spec(b) is None]
if chybaju:
    print("inštalujem:", chybaju)
    !{sys.executable} -m pip install --no-cache-dir -q -r requirements_e4.txt
else:
    print("všetky balíčky už sú nainštalované")

**Bunka 3:** GPU a bitsandbytes. Musí vypísať verzie a `bitsandbytes: OK`.

In [ ]:
import torch, transformers, bitsandbytes as bnb

print("torch", torch.__version__, "| transformers", transformers.__version__, "| bitsandbytes", bnb.__version__)
assert torch.cuda.is_available(), "torch nevidí GPU: je zvolená konfigurácia s GPU?"
print("GPU:", torch.cuda.get_device_name(0), f"| voľná pamäť {torch.cuda.mem_get_info()[0] / 1024**3:.1f} GB")
# malá 4-bitová vrstva na GPU: overí, že bitsandbytes má funkčné jadrá pre túto kartu
vrstva = bnb.nn.Linear4bit(256, 256, compute_dtype=torch.float16, quant_type="nf4").to("cuda")
y = vrstva(torch.randn(4, 256, device="cuda", dtype=torch.float16))
assert torch.isfinite(y).all()
del vrstva, y
torch.cuda.empty_cache()
print("bitsandbytes: OK")

**Bunka 4:** stiahne 7B model do `MODEL_CESTA` (asi 5,5 GB). Ak skončí chybou, že repozitár neexistuje, v bunke 1 prepni `MODEL` na zálohu (15 GB) a spusti bunky 1 a 4 znova.

In [ ]:
from huggingface_hub import snapshot_download

t = time.time()
snapshot_download(MODEL, local_dir=MODEL_CESTA,
                  allow_patterns=["*.json", "*.safetensors", "*.txt", "tokenizer*", "LICENSE*"])
velkost = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(MODEL_CESTA) for f in fs)
print(f"model v {MODEL_CESTA}: {velkost / 1024**3:.1f} GB, trvalo {time.time() - t:.0f} s")

**Bunka 5:** skúška na prvých 20 správach (40 otázok). Overí, že sa 4-bitový model načíta a beží, a ukáže rýchlosť. Čas na riadku `20/20 správ (… s)` zahŕňa aj načítanie modelu (riadok `model načítaný za … s`); ich rozdiel delený 40 je čas na jednu otázku. Celý valid má 1 130 otázok.

Ak beh skončí chybou `NaN/inf` alebo ak vychádza viac ako 2 s na otázku (celý valid by trval vyše 40 minút), zastav sa a pošli mi výpis.

In [ ]:
!{sys.executable} src/e4_llm.py --backend hf --model {MODEL} --model-cesta {MODEL_CESTA} --4bit --limit 20

**Bunka 6:** prompt v1 na celom valid (rovnaký ako pri 3B, kvôli priamemu porovnaniu). Prvých 20 správ sa vezme z uložených odpovedí.

In [ ]:
!{sys.executable} src/e4_llm.py --backend hf --model {MODEL} --model-cesta {MODEL_CESTA} --4bit

**Bunka 7:** prompt v2 (počty a príklady z train) na celom valid.

In [ ]:
!{sys.executable} src/e4_llm.py --backend hf --model {MODEL} --model-cesta {MODEL_CESTA} --4bit --prompt v2

**Bunka 8:** zbalí výsledky 7B do `vysledky_e4c.zip` v koreni projektu (`NLP_food_hazard/`). Stiahni ho a vlož do koreňa projektu na svojom PC.

In [ ]:
subory = sorted(glob.glob("results/etapa2/llm_cache/*7B*") + glob.glob("results/etapa2/llm/detail_*7B*")
                + glob.glob("results/etapa2/predikcie/E4_*7B*") + glob.glob("results/etapa2/logy/E4_*7B*"))
with zipfile.ZipFile("vysledky_e4c.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for s in subory:
        z.write(s)
        print(s)
print(f"\n{len(subory)} súborov v {os.path.join(KOREN, 'vysledky_e4c.zip')}")

**Bunka 9 (až na konci projektu):** zmaže len 7B model (3B ostane). Odkomentuj a spusti.

In [ ]:
# shutil.rmtree(MODEL_CESTA)
# print("zmazané:", MODEL_CESTA)